# Beginner 06 — Agent Identity Lifecycle

We will build a small enterprise **Agent Identity Registry**.

The lab focuses on lifecycle semantics rather than a vendor-specific IAM API. Later courses connect these concepts to SPIFFE/SPIRE, OAuth, Entra Agent ID, policy engines, and cloud identity.


In [ ]:
from dataclasses import dataclass, field, asdict
from datetime import datetime, timedelta, timezone
from enum import Enum
from typing import Optional
import json, uuid

def now():
    return datetime.now(timezone.utc)


## 1 — Explicit lifecycle states

In [ ]:
class State(str, Enum):
    DRAFT = "draft"
    REGISTERED = "registered"
    UNDER_REVIEW = "under_review"
    APPROVED = "approved"
    PROVISIONED = "provisioned"
    ACTIVE = "active"
    SUSPENDED = "suspended"
    REVOKED = "revoked"
    RETIRING = "retiring"
    RETIRED = "retired"
    REJECTED = "rejected"

ALLOWED_TRANSITIONS = {
    State.DRAFT: {State.REGISTERED},
    State.REGISTERED: {State.UNDER_REVIEW},
    State.UNDER_REVIEW: {State.APPROVED, State.REJECTED},
    State.APPROVED: {State.PROVISIONED},
    State.PROVISIONED: {State.ACTIVE},
    State.ACTIVE: {State.SUSPENDED, State.REVOKED, State.RETIRING},
    State.SUSPENDED: {State.ACTIVE, State.REVOKED, State.RETIRING},
    State.REVOKED: {State.RETIRING},
    State.RETIRING: {State.RETIRED},
    State.REJECTED: set(),
    State.RETIRED: set(),
}


## 2 — Registry record

In [ ]:
@dataclass
class AgentRecord:
    agent_id: str
    name: str
    purpose: str
    sponsor: str
    technical_owner: str
    risk_tier: str
    environment: str
    state: State = State.DRAFT
    autonomy: str = "bounded"
    workload_ids: set[str] = field(default_factory=set)
    tools: set[str] = field(default_factory=set)
    credentials: dict[str, dict] = field(default_factory=dict)
    access_grants: dict[str, dict] = field(default_factory=dict)
    created_at: datetime = field(default_factory=now)
    updated_at: datetime = field(default_factory=now)
    last_reviewed_at: Optional[datetime] = None
    next_review_at: Optional[datetime] = None
    version: str = "1.0.0"


## 3 — Lifecycle event log

In [ ]:
EVENTS = []

def emit(agent, event_type, actor, details=None):
    event = {
        "event_id": str(uuid.uuid4()),
        "timestamp": now().isoformat(),
        "agent_id": agent.agent_id,
        "type": event_type,
        "actor": actor,
        "details": details or {},
    }
    EVENTS.append(event)
    return event


## 4 — Registry and controlled transitions

In [ ]:
class Registry:
    def __init__(self):
        self.agents = {}

    def create(self, **kwargs):
        agent = AgentRecord(
            agent_id=f"agent:{uuid.uuid4().hex[:10]}",
            **kwargs,
        )
        self.agents[agent.agent_id] = agent
        emit(agent, "agent.created", kwargs["technical_owner"])
        return agent

    def transition(self, agent_id, new_state, actor, reason):
        agent = self.agents[agent_id]
        if new_state not in ALLOWED_TRANSITIONS[agent.state]:
            raise ValueError(f"illegal transition: {agent.state} -> {new_state}")
        old = agent.state
        agent.state = new_state
        agent.updated_at = now()
        emit(agent, "agent.state_changed", actor, {
            "from": old.value, "to": new_state.value, "reason": reason
        })
        return agent

registry = Registry()

agent = registry.create(
    name="Travel Booking Agent",
    purpose="Book policy-compliant employee travel",
    sponsor="user:alice",
    technical_owner="team:travel-platform",
    risk_tier="high",
    environment="prod",
)
print(agent.agent_id, agent.state)


## 5 — Registration and review

In [ ]:
registry.transition(agent.agent_id, State.REGISTERED,
                    "team:travel-platform", "metadata complete")
registry.transition(agent.agent_id, State.UNDER_REVIEW,
                    "iam:workflow", "begin risk/security review")
registry.transition(agent.agent_id, State.APPROVED,
                    "user:risk-approver", "controls accepted")
print(agent.state)


## 6 — Provision runtime bindings, tools, and credential profiles

In [ ]:
def provision(agent, *, workload_ids, tools, credential_profiles, actor):
    if agent.state != State.APPROVED:
        raise PermissionError("agent must be approved before provisioning")

    agent.workload_ids.update(workload_ids)
    agent.tools.update(tools)

    for profile in credential_profiles:
        agent.credentials[profile] = {
            "status": "configured",
            "generation": 0,
            "last_rotated": None,
        }

    emit(agent, "agent.provisioned_assets", actor, {
        "workload_ids": sorted(workload_ids),
        "tools": sorted(tools),
        "credential_profiles": credential_profiles,
    })

provision(
    agent,
    workload_ids={"spiffe://corp.example/prod/travel-booking"},
    tools={"flight.search", "flight.book"},
    credential_profiles=["travel-search", "travel-book-limited"],
    actor="iam:provisioner",
)

registry.transition(agent.agent_id, State.PROVISIONED,
                    "iam:provisioner", "identity assets provisioned")


## 7 — Activation invariants

In [ ]:
def activation_errors(agent):
    errors = []
    if not agent.sponsor:
        errors.append("missing sponsor")
    if not agent.technical_owner:
        errors.append("missing technical owner")
    if not agent.workload_ids:
        errors.append("no approved workload identity")
    if not agent.credentials:
        errors.append("no credential profile")
    if agent.risk_tier not in {"low", "medium", "high", "critical"}:
        errors.append("invalid risk tier")
    return errors

errors = activation_errors(agent)
assert not errors, errors

registry.transition(agent.agent_id, State.ACTIVE,
                    "platform:deployment", "production activation")
print(agent.state)


## 8 — Credential rotation

In [ ]:
def rotate_credential(agent, profile, actor):
    if agent.state not in {State.ACTIVE, State.SUSPENDED}:
        raise PermissionError("identity not in rotatable state")
    record = agent.credentials[profile]
    record["generation"] += 1
    record["last_rotated"] = now().isoformat()
    emit(agent, "credential.rotated", actor, {
        "profile": profile,
        "generation": record["generation"],
    })

rotate_credential(agent, "travel-search", "workload-identity:controller")
rotate_credential(agent, "travel-search", "workload-identity:controller")
print(agent.credentials["travel-search"])
print("Logical identity unchanged:", agent.agent_id)


## 9 — Time-bound access grants

In [ ]:
def grant_access(agent, grant_id, resource, permissions, expires_at, actor):
    agent.access_grants[grant_id] = {
        "resource": resource,
        "permissions": list(permissions),
        "expires_at": expires_at,
        "status": "active",
    }
    emit(agent, "access.granted", actor, {
        "grant_id": grant_id,
        "resource": resource,
        "expires_at": expires_at.isoformat(),
    })

grant_access(
    agent,
    "grant:travel-prod",
    "travel-api:prod",
    ["search", "book_limited"],
    now() + timedelta(days=90),
    "iam:entitlement-workflow",
)


## 10 — Risk-based review cadence

In [ ]:
REVIEW_DAYS = {
    "low": 365,
    "medium": 180,
    "high": 90,
    "critical": 30,
}

def complete_review(agent, reviewer, decision="continue"):
    agent.last_reviewed_at = now()
    agent.next_review_at = now() + timedelta(days=REVIEW_DAYS[agent.risk_tier])
    emit(agent, "review.completed", reviewer, {
        "decision": decision,
        "next_review_at": agent.next_review_at.isoformat(),
    })

complete_review(agent, "user:alice")
print("Next review:", agent.next_review_at)


## 11 — Detect expired grants

In [ ]:
def expire_access(agent, at=None):
    at = at or now()
    expired = []
    for grant_id, grant in agent.access_grants.items():
        if grant["status"] == "active" and at >= grant["expires_at"]:
            grant["status"] = "expired"
            expired.append(grant_id)
            emit(agent, "access.expired", "iam:lifecycle", {"grant_id": grant_id})
    return expired

future = now() + timedelta(days=100)
print("Expired:", expire_access(agent, future))


## 12 — Detect orphan conditions

In [ ]:
ACTIVE_IDENTITIES = {
    "user:alice": True,
    "team:travel-platform": True,
    "user:left-company": False,
}

def orphan_reasons(agent):
    reasons = []
    if not ACTIVE_IDENTITIES.get(agent.sponsor, False):
        reasons.append("sponsor missing or inactive")
    if not ACTIVE_IDENTITIES.get(agent.technical_owner, False):
        reasons.append("technical owner missing or inactive")
    return reasons

print(orphan_reasons(agent))


## 13 — Simulate sponsor departure and remediation

In [ ]:
ACTIVE_IDENTITIES["user:alice"] = False
print("Detected:", orphan_reasons(agent))

def transfer_sponsor(agent, new_sponsor, actor):
    old = agent.sponsor
    agent.sponsor = new_sponsor
    emit(agent, "sponsor.changed", actor, {"from": old, "to": new_sponsor})

ACTIVE_IDENTITIES["user:manager-alice"] = True
transfer_sponsor(agent, "user:manager-alice", "iam:lifecycle-workflow")
print("After remediation:", orphan_reasons(agent))


## 14 — Material change detection

In [ ]:
def classify_change(change):
    material_fields = {"tools", "data_classification", "autonomy", "purpose", "permissions"}
    if any(field in change for field in material_fields):
        return "material"
    return "minor"

changes = [
    {"version": "1.0.1", "bugfix": "logging"},
    {"version": "2.0.0", "tools": ["flight.search", "flight.book", "payment.refund"]},
]

for change in changes:
    print(change["version"], "->", classify_change(change))


A material change should trigger risk/access review instead of silently inheriting old approval.

## 15 — Suspension

In [ ]:
registry.transition(
    agent.agent_id,
    State.SUSPENDED,
    "security:soc",
    "anomalous booking behavior under investigation",
)
print(agent.state)


## 16 — Reactivation after investigation

In [ ]:
registry.transition(
    agent.agent_id,
    State.ACTIVE,
    "security:soc",
    "incident cleared and credentials rotated",
)
rotate_credential(agent, "travel-book-limited", "security:soc")
print(agent.state)


## 17 — Emergency revocation

In [ ]:
def emergency_revoke(registry, agent, actor, reason):
    if agent.state == State.ACTIVE:
        registry.transition(agent.agent_id, State.REVOKED, actor, reason)
    elif agent.state == State.SUSPENDED:
        registry.transition(agent.agent_id, State.REVOKED, actor, reason)
    else:
        raise PermissionError(f"cannot emergency revoke from {agent.state}")

    for credential in agent.credentials.values():
        credential["status"] = "revoked"

    for grant in agent.access_grants.values():
        if grant["status"] == "active":
            grant["status"] = "revoked"

    emit(agent, "agent.emergency_revocation_completed", actor, {
        "credentials_revoked": len(agent.credentials),
        "access_grants_processed": len(agent.access_grants),
    })

emergency_revoke(
    registry, agent, "security:soc",
    "suspected credential compromise",
)
print(agent.state)
print(agent.credentials)


## 18 — Revocation invariant

In [ ]:
def assert_revocation_invariants(agent):
    assert agent.state == State.REVOKED
    assert all(c["status"] == "revoked" for c in agent.credentials.values())
    assert all(g["status"] != "active" for g in agent.access_grants.values())
    return True

print(assert_revocation_invariants(agent))


## 19 — Retirement and deprovisioning

In [ ]:
def deprovision(agent, actor):
    removed = {
        "workload_ids": sorted(agent.workload_ids),
        "tools": sorted(agent.tools),
        "credential_profiles": sorted(agent.credentials),
        "access_grants": sorted(agent.access_grants),
    }
    agent.workload_ids.clear()
    agent.tools.clear()
    agent.credentials.clear()
    agent.access_grants.clear()
    emit(agent, "agent.deprovisioned", actor, removed)
    return removed

registry.transition(agent.agent_id, State.RETIRING,
                    "user:manager-alice", "agent replaced by new platform")
removed = deprovision(agent, "iam:deprovisioner")
registry.transition(agent.agent_id, State.RETIRED,
                    "iam:deprovisioner", "downstream access removed")

print(agent.state)
print(json.dumps(removed, indent=2))


## 20 — Retired identities cannot reactivate directly

In [ ]:
try:
    registry.transition(agent.agent_id, State.ACTIVE,
                        "developer", "bring it back")
except ValueError as e:
    print("DENIED:", e)


## 21 — Audit timeline

In [ ]:
for event in EVENTS:
    print(
        event["timestamp"],
        event["type"],
        event["actor"],
        event["details"],
    )


## 22 — Lifecycle metrics

In [ ]:
def lifecycle_metrics(registry):
    agents = list(registry.agents.values())
    return {
        "total_agents": len(agents),
        "active": sum(a.state == State.ACTIVE for a in agents),
        "suspended": sum(a.state == State.SUSPENDED for a in agents),
        "retired": sum(a.state == State.RETIRED for a in agents),
        "missing_sponsor": sum(not a.sponsor for a in agents),
        "missing_owner": sum(not a.technical_owner for a in agents),
        "overdue_review": sum(
            a.state == State.ACTIVE
            and a.next_review_at is not None
            and now() > a.next_review_at
            for a in agents
        ),
    }

print(json.dumps(lifecycle_metrics(registry), indent=2))


## 23 — Exercise: build an onboarding workflow

Create a second agent:

```text
Claims Settlement Agent
risk = critical
autonomy = high
```

Requirements:

1. business sponsor;
2. technical owner;
3. security approval;
4. finance approval;
5. approved production workload identity;
6. no permanent credentials;
7. 30-day formal access review;
8. refunds over CAD 500 require human approval;
9. material tool changes force re-review.

Implement the lifecycle from DRAFT to ACTIVE.

## 24 — Exercise: orphan workflow

Implement:

```python
reconcile_ownership(registry, directory)
```

Behavior:

```text
inactive technical owner -> notify platform team
inactive sponsor -> transfer to manager if known
no replacement sponsor -> suspend high-risk agent
```

Emit evidence for every automated decision.

## 25 — Exercise: revocation propagation

Model:

```text
registry
authorization cache
credential
running session
tool gateway
```

Emergency revoke an agent and calculate:

```text
time until every layer rejects it
```

Define an enterprise SLO such as:

```text
critical-agent revocation propagation < 60 seconds
```

Then identify which component prevents meeting the SLO.

## 26 — Exercise: lifecycle policy as tests

Write assertions for:

```text
ACTIVE requires sponsor
ACTIVE requires owner
ACTIVE requires workload identity
HIGH risk requires review <= 90 days
CRITICAL risk requires review <= 30 days
REVOKED has no active grants
RETIRED has no credentials
RETIRED cannot become ACTIVE
```

This is an introduction to **governance as code**.

## Review questions

1. Why are logical agent, workload, and credential lifecycles separate?
2. Why should every agent have a registry record?
3. What is the difference between owner and sponsor?
4. How do orphaned agents arise?
5. Why should activation have explicit invariants?
6. Why should credentials rotate without changing logical identity?
7. What kinds of changes should trigger re-review?
8. Why are expiring grants preferable to standing access?
9. How does suspension differ from revocation?
10. Why must revocation propagate beyond the registry?
11. What must be removed during retirement?
12. Which lifecycle evidence should be retained?

# Next course

## Intermediate 01 — Workload Identity with SPIFFE & SPIRE
